In [ ]:
%pip install torch torchvision xgboost scikit-learn opencv-python numpy matplotlib seaborn joblib

In [ ]:
import os
import random
import numpy as np
import cv2
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from PIL import Image

from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

root = "dataset/"
for folder in os.listdir(root):
    path = os.path.join(root, folder)
    print(folder, ":", len(os.listdir(path)), "images")

dataset = datasets.ImageFolder(root)
print(dataset.class_to_idx)

counts = [len(os.listdir(os.path.join(root, cls))) for cls in dataset.classes]
bars = plt.bar(dataset.classes, counts)
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5, str(count), ha='center')
plt.xticks(rotation=45)
plt.ylabel("Image Count")
plt.title("Training Set Class Distribution")
plt.tight_layout()
plt.show()

In [ ]:
torch.manual_seed(42)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder("dataset/", transform=transform)

train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print("Images leak:", len(set(train_dataset.indices).intersection(set(test_dataset.indices))))
print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

In [ ]:
# ── CNN Backbone (CNN features only — 1280 dimensions) ──
mobilenet = models.mobilenet_v2(pretrained=True).to(device)
mobilenet.classifier = nn.Identity()
for param in mobilenet.parameters():
    param.requires_grad = False
mobilenet.eval()

def feature_extraction(loader, model):
    all_features, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            cnn_feat = model(images).cpu()   # shape: (batch, 1280)
            all_features.append(cnn_feat)
            all_labels.append(labels)
    return torch.cat(all_features), torch.cat(all_labels)

print("Extracting features...")
train_features, train_labels = feature_extraction(train_loader, mobilenet)
test_features,  test_labels  = feature_extraction(test_loader,  mobilenet)

X_train, y_train = train_features.numpy(), train_labels.numpy()
X_test,  y_test  = test_features.numpy(),  test_labels.numpy()

print("Train features shape:", X_train.shape)
print("Test  features shape:", X_test.shape)

## Feature Extraction Visualization
Shows how MobileNetV2 processes an image across different layers.

In [ ]:
# ── Multi-layer CNN Feature Visualization ──
sample_path, sample_label = random.choice(dataset.samples)
class_name = dataset.classes[sample_label]

pil_img = Image.open(sample_path).convert("RGB")
img_tensor = transform(pil_img).unsqueeze(0).to(device)

# Layers to visualize: early, middle, deep
layer_indices = [2, 9, 18]
layer_names   = ["Early Layer 2\n(edges & colors)", "Middle Layer 9\n(shapes & textures)", "Deep Layer 18\n(object features)"]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Original image
axes[0].imshow(pil_img)
axes[0].set_title(f"Original Image\n({class_name})", fontsize=11)
axes[0].axis("off")

# Feature maps at each layer
for i, (idx, name) in enumerate(zip(layer_indices, layer_names)):
    with torch.no_grad():
        conv = torch.nn.Sequential(*list(mobilenet.features.children())[:idx])
        fmap = conv(img_tensor).cpu()

    heatmap = fmap[0].mean(dim=0).numpy()
    heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

    axes[i+1].imshow(heatmap, cmap="jet")
    axes[i+1].set_title(f"{name}", fontsize=11)
    axes[i+1].axis("off")

plt.suptitle("MobileNetV2 Feature Maps Across Layers", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Sample class: {class_name}")
print("Early layers detect simple patterns like edges and colors")
print("Middle layers detect shapes and textures")
print("Deep layers detect high-level object features used for classification")

## Model Registry

To add a new model, just add one entry to the `MODELS` dict below and re-run from this cell onwards.

In [ ]:
# ══════════════════════════════════════════════════
# ADD / REMOVE MODELS HERE
# ══════════════════════════════════════════════════
MODELS = {
    "XGBoost": XGBClassifier(
        n_estimators=350,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        use_label_encoder=False,
        eval_metric="mlogloss"
    ),
    "Soft Voting": VotingClassifier(
        estimators=[
            ("xgb", XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric="mlogloss")),
            ("rf",  RandomForestClassifier(n_estimators=100, random_state=42)),
            ("lr",  LogisticRegression(max_iter=5000, random_state=42)),
        ],
        voting="soft"
    ),
    "Stacking": StackingClassifier(
        estimators=[
            ("knn", KNeighborsClassifier(n_neighbors=5)),
            ("rf",  RandomForestClassifier(n_estimators=100, random_state=42)),
            ("dt",  DecisionTreeClassifier(max_depth=10, random_state=42)),
        ],
        final_estimator=LogisticRegression(max_iter=5000),
        cv=5
    ),
    "Bagging": BaggingClassifier(
        estimator=DecisionTreeClassifier(),
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
}
print(f"Will train {len(MODELS)} models:", list(MODELS.keys()))

In [ ]:
results = {}

for name, clf in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print('='*50)
    clf.fit(X_train, y_train)
    preds  = clf.predict(X_test)
    acc    = accuracy_score(y_test, preds)
    report = classification_report(y_test, preds, target_names=dataset.classes, output_dict=True)
    results[name] = {"preds": preds, "accuracy": acc, "report": report, "clf": clf}
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds, target_names=dataset.classes))

print("\n✅ All models trained!")

In [ ]:
# ── Accuracy Comparison ──
names = list(results.keys())
accs  = [results[n]["accuracy"] for n in names]

colors = plt.cm.tab10.colors
plt.figure(figsize=(10, 5))
bars = plt.bar(names, accs, color=colors[:len(names)])
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{acc:.3f}", ha="center", fontsize=10)
plt.ylim(0, 1.05)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Accuracy")
plt.title("Model Accuracy Comparison")
plt.tight_layout()
plt.show()

print(f"\n{'Model':<25} {'Accuracy':>10}")
print("-" * 37)
for name, acc in sorted(zip(names, accs), key=lambda x: -x[1]):
    print(f"{name:<25} {acc:>10.4f}")

In [ ]:
# ── F1 Score per Class ──
n_models  = len(results)
n_classes = len(dataset.classes)
x     = np.arange(n_classes)
width = 0.8 / n_models

fig, ax = plt.subplots(figsize=(12, 5))
for i, (name, res) in enumerate(results.items()):
    f1s = [res["report"][cls]["f1-score"] for cls in dataset.classes]
    ax.bar(x + i * width - 0.4 + width/2, f1s, width, label=name)

ax.set_xticks(x)
ax.set_xticklabels(dataset.classes, rotation=45, ha="right")
ax.set_ylim(0, 1.1)
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score per Class — All Models")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrices ──
n    = len(results)
cols = min(3, n)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
axes = np.array(axes).flatten() if n > 1 else [axes]

for ax, (name, res) in zip(axes, results.items()):
    cm      = confusion_matrix(y_test, res["preds"])
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=dataset.classes,
                yticklabels=dataset.classes, ax=ax)
    ax.set_title(f"{name} (acc={res['accuracy']:.3f})")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

for ax in axes[n:]:
    ax.set_visible(False)

plt.suptitle("Normalized Confusion Matrices", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Prediction Distribution ──
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
axes = np.array(axes).flatten() if n > 1 else [axes]

for ax, (name, res) in zip(axes, results.items()):
    pred_counts = np.bincount(res["preds"], minlength=len(dataset.classes))
    ax.bar(dataset.classes, pred_counts, color="orange")
    ax.set_title(f"{name} — Prediction Distribution")
    ax.set_xticklabels(dataset.classes, rotation=45, ha="right")
    ax.set_ylabel("Count")

for ax in axes[n:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── Save Best Model ──
best_name = max(results, key=lambda n: results[n]["accuracy"])
best_clf  = results[best_name]["clf"]

os.makedirs("model", exist_ok=True)
joblib.dump(best_clf,        "model/classifier.pkl")
joblib.dump(dataset.classes, "model/classes.pkl")

print(f"✅ Saved best model: {best_name}")
print(f"   Accuracy: {results[best_name]['accuracy']:.4f}")
print(f"   Saved to: model/classifier.pkl")

# To save a specific model instead:
# joblib.dump(results["XGBoost"]["clf"], "model/classifier.pkl")